In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   
model.train(
    data="cart.yaml",
    epochs=50,
    imgsz=640,
)


In [ ]:
from deep_sort_realtime.deepsort_tracker import DeepSort
import cv2 
import pandas as pd 
import csv
import numpy as np 
import time 

def inside_cart(cart_bbox , bbox  ) : 
    x1 , y1 , x2 , y2 = cart_bbox
    a  = bbox[0] 
    b = bbox[1]
    if a >=  x1 and a <= x2 and b <= y2 and b >= y1 : 
        print('Not Shoplifted') 
        return 1 
    return 0 


with open('shoplifted_objects.csv' , 'w') as new_file  : 
        feild  = ['human_id' , 'item_id'] 
        csv_writer = csv.DictWriter(new_file , feild names  = feild)
        csv_writer.writeheader()


trigger  = 5400 
Tracker   = DeepSort(moved_objects , frame = frame )
flag = 0 
counter  = 0 
webcam  = cv2.VideoCapture(0) 
updated_id = []
hash = {}
initial_id = []
intial_object_list = []
updated_object_list = []
shoplifted_parameters  = []
shoplifted = []
shoplifted_id = []
df = pd.read_csv('data.csv')
df['center_x'] = (df['bbox1'] + df['bbox3'])/2
df['center_y'] = (df['bbox2'] + df['bbox4'])/2
while True: 
    ret , frame = webcam.imread() 
    tracks  = Tracker.update_tracks() 
    results = model("frame")
    cart_bbox = results[0].boxes.xyxy
    if initial_id.size == 0 :  
        for box in tracks : 
            id = box.rack_id
            initial_id.append(id)
            intial_object_list.append([box , id]) 
    else :  
        updated_id = []
        for box in tracks : 
            id = box.rack_id
            updated_id.append(id)
            updated_object_list.append([box , id])
        missing = set(updated_id) - set(initial_id)
    if len(missing) != 0 :
        for i in missing : 
            if hash[i] < trigger : 
                hash[i] = hash[i] + 1
            else : 
                for k in intial_object_list : 
                    if k[2] == i :
                        x = k[0]
                        y = k[1]
                        shoplifted_parameters.append([x,y])
        shoplifted_parameters = np.array(shoplifted_parameters)        
        for x , y in shoplifted_parameters :
            shoplifted = df[abs(df['center_x'] - x <= 1.0 ) and abs(df['center_y'] - y )] 
            shoplifted_id.append(shoplifted['id'])
        for j in updated_id :   
            hash[j] = 0